# Datamine LUNICOND Process Example

This notebook demonstrates how to configure and run the **`lunicond`** process wrapper in `dmstudio`.

## Process Description

## LUNICOND

The goal of Localized Uniform Conditioning is to provide a more 'intuitive' presentation of uniform conditioning result by assigning a unit-level mean average grade based on calculations of the tonnages and grades above each cut-off grade, dispersed within the panel according to their rank. This process is implemented interactively via the Uniform Conditioning Wizard.

### Input Files:

* **ucmodel** (Model):
A model file containing the conditioned panels. This model is the output model from the
process **UNIFCOND** , containing the uniform conditioned distribution (metal, grade and
proportion at each cut-off).
Required=Yes

* **smumodel** (Model):
An SMU model, containing a linear estimate of a grade, for the localization ranking of
the uniform conditioned panel distribution.
Required=No

### Output Files:

* **outmodel** (Undefined):
The output locally-conditioned model.
Required=Yes

### Fields:

* **kriging** (Alphanumeric : IN):
The field in the input SMU model containing the linear estimated values upon which the
UC distribution will be ranked.
Default=Undefined
Required=Yes

* **outfield** (Alphanumeric : IN):
The field in the output locally-conditioned model (**OUTMODEL**) to contain the values
resulting from localized uniform conditioning.
Default=Undefined
Required=Yes

### Parameters:

* **cutmin**:
The minimum cutoff grade to be considered during Uniform Conditioning.
Range=
Values=
Default=0
Required=No

* **cutint**:
The size of each grade cutoff interval
Range=
Values=
Default=10
Required=No

* **cutnum**:
The number of grade cutoff intervals to considered during Uniform Conditioning.
Range=
Values=
Default=10
Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('lunicond')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute lunicond
print("Running lunicond...")
dm_cmd.lunicond(
    ucmodel_i='t_mod1',  # required
    smumodel_i='t_mod1',  # required
    outmodel_o='t_lunicond_out',  # required
    kriging_f='optional',  # required
    outfield_f='optional',  # required
    # cutmin_p=0,  # optional
    # cutint_p=10,  # optional
    # cutnum_p=10,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("lunicond execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_lunicond_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")